# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begs44/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

* **Task Type:** Ranking / Scoring (Binary Classification Proxy)
* **Why this task type:** Editorial teams have limited weekly capacity (20–50 URLs) out of 30,000 pages. They do not need an isolated label for every URL; they need an ordered triage list ranked by probability of organic decay ($P(\text{decay}=1)$) so they can audit from highest risk downward.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/begs44/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Observed distribution of candidate signals for ranking
print(df["trend_direction"].value_counts(normalize=True).round(3))

trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

* **Target / Proxy:** Organic traffic decay proxy (`target_decay`).
* **Source of Label:** Observed outcome from Google performance data (`trend_direction == 'down'`). True revenue loss cannot be observed before review, so measured impression/ranking decline serves as the operational proxy.

In [ ]:
# Constructing binary target proxy
df["target_decay"] = (df["trend_direction"] == "down").astype(int)

print(f"Target count (1 = decaying): {df['target_decay'].sum():,} / {len(df):,}")
print(f"Base decay rate: {df['target_decay'].mean():.1%}")


## 3. Success metric

*One metric you can defend. What number means 'good'?*

* **Defensible Metric:** Precision@50 (evaluating top 50 ranked URLs against editorial batch capacity).
* **What means 'good':** $\ge 0.70$ (at least 35 of the top 50 flagged pages are genuinely decaying). The random baseline is 0.542; achieving $\ge 0.70$ delivers a meaningful lift without wasting editorial hours on healthy URLs.

In [ ]:
# Base rate vs minimum operational threshold
baseline_precision = df["target_decay"].mean()
target_threshold = 0.70

print(f"Random Baseline Precision: {baseline_precision:.3f}")
print(f"Target 'Good' Precision@50: {target_threshold:.2f}")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

* **Unit of Analysis:** One row = One unique published URL (`content_id`).
* **Pre-decision Signals:** `impressions_90d`, `avg_position`, `ctr`, `content_age_days`.

In [ ]:
# Unit of analysis check and dataframe slice
cols = ["content_id", "impressions_90d", "avg_position", "ctr", "content_age_days", "target_decay"]
df[cols].head(5)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

* **Search Volume Decoupling:** Keyword search volume has zero linear correlation with actual impressions ($r = 0.001$), so fixed volume thresholds fail.
* **Non-linear Signal Trade-offs:** High-impression pages dropping 2 positions lose more traffic than zero-impression pages dropping 15 positions; simple `if-else` rules cannot balance magnitude against rank.
* **Calibrated Output:** Static rules yield either thousands of unranked URLs or zero results. ML provides a continuous priority curve fitted to weekly editorial hours.

In [ ]:
# Showing limitations of a static heuristic rule (e.g. age > 180 and position > 15)
heuristic = (df["content_age_days"] > 180) & (df["avg_position"] > 15)
rule_precision = df.loc[heuristic, "target_decay"].mean()

print(f"Flagged by rule: {heuristic.sum():,} rows")
print(f"Rule Precision: {rule_precision:.3f} vs Base Rate: {df['target_decay'].mean():.3f}")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.